<a href="https://colab.research.google.com/github/ShravaniV26/ML-Lab-programs/blob/main/FOIL_%26_RIPPER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install wittgenstein pandas scikit-learn

import pandas as pd
from sklearn.datasets import load_iris
import wittgenstein as lw

iris = load_iris(as_frame=True)
X = iris.data
y = iris.target

df = pd.concat([X, y.rename("target")], axis=1)
print("Dataset head:\n", df.head())

model = lw.RIPPER()
model.fit(X, y, pos_class=0)

print("\n=== RIPPER Rules ===")
print(model.ruleset_)

df_bin = df.copy()
df_bin['target'] = (df_bin['target'] == 0).astype(int) # 1 =setosa, 0 = others

attributes = list(X.columns)

def foil_gain(pos_before, neg_before, pos_after, neg_after):
   if pos_after == 0: return-1e9
   return pos_after * ( ((pos_after)/(pos_after+neg_after))- ((pos_before)/(pos_before+neg_before)) )


def foil(df, target_col='target'):
  rules = []
  pos_total = df[target_col].sum()
  neg_total = len(df)- pos_total

  while pos_total > 0:
    rule = []
    pos_rem, neg_rem = pos_total, neg_total
    covered = df.copy()

    while neg_rem > 0:
      best_gain, best_attr =-1e9, None
      for attr in attributes:
        for val in df[attr].unique():
          subset = covered[covered[attr] == val]
          pos_after = subset[target_col].sum()
          neg_after = len(subset)- pos_after
          gain = foil_gain(pos_rem, neg_rem, pos_after,neg_after)
          if gain > best_gain:
            best_gain, best_attr, best_val,best_subset = gain, attr, val, subset

      if best_attr is None: break
      rule.append((best_attr, best_val))
      covered = best_subset
      pos_rem, neg_rem = covered[target_col].sum(), len(covered)- covered[target_col].sum()
    rules.append((rule, 1)) # predict positive
    df = df.drop(covered.index) # remove covered examples
    pos_total = df[target_col].sum()
    neg_total = len(df)- pos_total
  return rules

rules = foil(df_bin)
print("\n=== FOIL Learned Rules (Setosa vs Not) ===")
for conds, pred in rules:
  cond_str = " AND ".join([f"{a}={v}" for a,v in conds])
  print(f"IF {cond_str} THEN class={pred}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 5.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for wittgenstein: filename=wittgenstein-0.3.5-py3-none-any.whl size=98989 sha256=639d8da9b73f951825e0497f79ea0ba542143f35d6453a593913be6078dd7aeb
  Stored in directory: /root/.cache/pip/wheels/03/8e/f7/4afc64184996e67ef74ebefa7d74b324534a079fc3462242aa
Successfully built wittgenstein
Dataset head:
    sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0                5.1               3.5                1.4               0.2   
1                4.9               3.0                1.4               0.2   
2                4.7               3.2                1.3               0.2   
3                4.6               3.1                1.5               0.2   
4                5.0               3.6                1.4               0.2   

   target  
0       0  
1       0  
2       0  
3       0  
4       0  

=== RIPPER Rule